# 데이터 전처리 — 누락 데이터 · groupby · 이상치

수업 중 작성한 필기를 **실행 가능하도록 정리한 노트**입니다. (교재 `데이터분석` p.22~33)
각 단원은 `개념 설명 → 예제 코드` 순서로 구성되어 있고,
원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.

> 실습 데이터 `ad_performance.csv` · `weather.csv` 는 이 노트북과 **같은 폴더**에 있어야 합니다.
> 수업에서 쓴 도서 대출 데이터(`ns_202104.csv`, 401,682행)는 용량이 커서 저장소에 올리지 않았습니다.
> 1단원은 같은 상황을 **작은 표로 만들어** 재현했고, 원본 파일이 있으면 그것으로도 돌아가게 해 두었습니다.

## 0. 데이터 전처리(preprocessing)란

수집한 데이터를 **분석에 적합한 형태로 만들어 신뢰도를 높이는 작업**이다.
오늘 다루는 세 가지는 전처리에서 가장 자주 만나는 문제들이다.

| 문제 | 도구 |
|------|------|
| 값이 **비어 있다** (누락 데이터) | `isnull()` · `dropna()` · `fillna()` |
| 같은 값이 **여러 행에 반복된다** | `groupby()` · `agg()` |
| 유독 **작거나 큰 값**이 섞여 있다 (이상치) | `quantile()` · IQR · `clip()` |

> 표에 빈자리가 남아 있으면 **머신러닝 단계에서 오류가 난다.** 그래서 먼저 손봐야 한다.

## 1. 누락 데이터 처리하기 (교재 p.23~26)

**누락 데이터(missing value)** = 값이 비어 있는 데이터.

부르는 이름이 셋이다.

| 용어 | 쓰이는 곳 |
|------|-----------|
| `NaN` | `read_csv()` 등으로 파일을 DataFrame·Series 로 바꿀 때 **누락을 표현하는 값** |
| `null` · `na` | **함수 이름**에 쓰인다 (`isnull()`, `dropna()`, `fillna()`) |

In [ ]:
import pandas as pd
import numpy as np

# 수업에서 쓴 도서 대출 데이터와 같은 모양을 작게 만들어 본다
sample = pd.DataFrame({
    "번호":      [1, 2, 3, 4, 5],
    "도서명":    ["파이썬 입문", "데이터 분석", "통계의 힘", "머신러닝", "SQL 기초"],
    "저자":      ["김진호", np.nan, "박서준", "최유리", np.nan],
    "세트 ISBN": [np.nan, np.nan, "978-89-1", np.nan, "978-89-2"],
    "부가기호":  [np.nan, "03560", np.nan, "03560", "03560"],
    "대출건수":  [12, 7, 30, 5, 18],
})

df = sample
df

### 1-1. 누락된 데이터의 개수 확인 — `df.isnull().sum()` (교재 p.24)

- `isnull()` : 누락 여부를 확인해 **bool 형식**으로 돌려준다 (비었으면 `True`)
- `sum()` : bool 을 더하면 `True` 가 1로 계산되므로 **개수**가 된다

> ⚠️ 반복문으로 하나씩 세면 40만 행 × 13열 = **520만 번** 돌게 된다.
> 판다스는 이걸 **한 줄로, 훨씬 빠르게** 해 준다.

In [ ]:
print("[컬럼별 누락 개수]")
print(df.isnull().sum())

print("-" * 40)

# 전체 누락 개수 : sum() 을 두 번 (컬럼별 합계 -> 그 합계의 합계)
print("전체 누락 개수 :", df.isnull().sum().sum())

print("-" * 40)

# 누락 여부 자체를 보고 싶으면 (전체를 다 찍지 말고 앞부분만)
df.isnull().head()

> **📝 고친 부분**
> - `for` 문을 두 겹으로 돌려 `pd.isnull(value)` 를 하나씩 세던 것 → **`df.isnull().sum()`** 한 줄 (교재 p.24)
> - 변수명 `dfbool` → 담기는 값이 bool 이 아니라 **개수**라 이름이 맞지 않았다
> - `display(df.isnull())` 은 401,682행을 전부 찍는다 → `.head()`

### 1-2. 누락된 데이터 삭제 — `df.dropna()` (교재 p.25)

- 행 또는 열에 누락 데이터가 **1개만 있어도 그 행을 통째로 지운다**
- `ignore_index=True` : 지우고 남은 행의 **인덱스를 0부터 다시 매긴다**
- `subset=['컬럼1', '컬럼2']` : **지정한 컬럼**에 누락이 있는 행만 지운다

In [ ]:
# 전체 : 누락이 하나라도 있으면 그 행을 지운다
df_na1 = df.dropna(ignore_index=True)
print("원본", df.shape, "-> 전체 dropna", df_na1.shape)
print(df_na1)

print("-" * 40)

# 특정 컬럼만 : '부가기호'나 '저자'가 비어 있는 행만 지운다
df_na2 = df.dropna(subset=['부가기호', '저자'], ignore_index=True)
print("subset dropna", df_na2.shape)
print(df_na2)

### 1-3. 누락된 데이터 바꾸기 — `df.fillna()` (교재 p.26)

지우면 데이터가 줄어든다. 그래서 **적당한 값으로 채우는** 방법도 쓴다.

```python
df_cleaned = df.fillna("내용 없음")    # 문자열로 채우기
df_cleaned = df.fillna(0)             # 0으로 채우기
```

In [ ]:
df_na3 = df.fillna("NULL")        # "NULL" 말고 다른 문자열도 된다
print(df_na3)

print("-" * 40)

# 컬럼마다 다른 값으로 채울 수도 있다
df_na4 = df.fillna({"저자": "저자 미상", "부가기호": 0, "세트 ISBN": "없음"})
print(df_na4)
print("채운 뒤 누락 개수 :", df_na4.isnull().sum().sum())

> **📝 참고**
> 수업에서 쓴 `ns_202104.csv` 가 같은 폴더에 있으면 아래 셀이 그 파일로 똑같은 과정을 실행합니다.
> 없으면 건너뜁니다. (`low_memory=False` 는 필기에서 났던 `DtypeWarning` 을 없애 준다)

In [ ]:
import os

if os.path.exists("./ns_202104.csv"):
    df_book = pd.read_csv("./ns_202104.csv", low_memory=False)   # 혼합 자료형 경고 방지
    print("모양 :", df_book.shape)
    print("전체 누락 개수 :", df_book.isnull().sum().sum())
    print(df_book.isnull().sum())
else:
    print("ns_202104.csv 가 없어 건너뜁니다. (위 작은 표 예제로 대신합니다)")

## 2. 중복되는 데이터를 그룹별로 모아서 처리하기 (교재 p.27~29)

`groupby()` 는 **중복된 값을 기준으로 데이터를 그룹으로 나누고**, 그룹마다 따로 계산한다.

```python
df.groupby(by=['컬럼', ...]).func()
```

- `by` : 중복 여부를 확인해 그룹으로 묶을 컬럼
- `func` : 그룹별로 적용할 함수 — `sum()` `mean()` `count()` …

> SQL 의 `GROUP BY` 와 같은 일을 한다.
> SQL 은 DB 안에서 집계까지만 하지만, 판다스로 가져오면 그대로 **머신러닝까지** 이어 갈 수 있다.

In [ ]:
path = "./ad_performance.csv"
df = pd.read_csv(path)

print("모양 :", df.shape)

# 교재 p.28 예시 : "어떤 채널의 평균 매출이 가장 높은가?"
channel_rev = df.groupby(by='Channel')['Revenue'].mean()
print("\n[채널별 평균 매출]")
print(channel_rev.round(0))

print("-" * 40)

# 성별 평균 비용
gender_cost = df.groupby(by='Gender')['Cost'].mean()
print("[성별 평균 비용]")
print(gender_cost.round(1))

### 2-1. `agg()` — 그룹별로 여러 함수를 한 번에 (교재 p.29)

```python
df.groupby('컬럼').agg(
    새로운_컬럼명 = ('원본 컬럼명', '집계 함수'),
    ...
)
```

"각 광고 채널별로 지출한 비용의 총합과 평균, 매출액의 총합과 평균은?"

In [ ]:
channel_sum = df.groupby('Channel').agg(
    total_cost      = ('Cost', 'sum'),
    average_cost    = ('Cost', 'mean'),
    total_revenue   = ('Revenue', 'sum'),
    average_revenue = ('Revenue', 'mean'),
)

print(channel_sum.round(1).to_string())

> **📝 고친 부분**
> - `ch = df.groupby(by='Gender')['Cost'].mean()` 을 계산해 놓고 출력을 주석으로 막아 둬서 결과가 보이지 않았다 → 살림
> - 교재 p.28 의 예시(**채널별 평균 매출**)를 먼저 두고, 성별 평균 비용을 그다음에 두어 순서를 맞춤
> - 주석 오타 `SQL cunt` → **`count`**
> - `display(channel_sum)` → 소수점이 지저분해 `round(1)` 후 `to_string()`

## 3. 이상치(outlier) 처리하기 (교재 p.30~33)

**이상치** = 수집된 데이터 중 아주 작거나 매우 큰 값.

### 사분위수와 IQR

순서대로 정렬된 데이터를 **4구간으로 균등하게 나눌 때 기준이 되는 수**가 사분위수다.

| 이름 | 위치 | 뜻 |
|------|------|-----|
| **Q1** | 25% | 아래에서 1/4 지점 |
| **Q2** | 50% | 가운데 값 (중앙값 · 미디언) |
| **Q3** | 75% | 아래에서 3/4 지점 |

- **IQR** (Interquartile Range) = **`Q3 - Q1`** — 가운데 절반이 퍼져 있는 폭
- 정상 범위 **최솟값** = `Q1 - (1.5 * IQR)`
- 정상 범위 **최댓값** = `Q3 + (1.5 * IQR)`
- **이 범위를 벗어난 값이 이상치**

분위수는 `quantile(q)` 로 구한다. `q` 에 원하는 백분위수를 넣는다. (`df.quantile(q)` / `df['col'].quantile(q)`)

In [ ]:
path = "./weather.csv"
df = pd.read_csv(path)          # 기상청 날씨 데이터 (temp 기온 · hum 습도 · atm 기압 · speed 풍속)

print("모양 :", df.shape)
print(df.head())

print("=" * 50)

q1 = df.quantile(q=0.25)        # 25% 지점
q3 = df.quantile(q=0.75)        # 75% 지점
iqr = q3 - q1                   # 사분위 범위

min_val = q1 - (1.5 * iqr)      # 정상 범위 최솟값
max_val = q3 + (1.5 * iqr)      # 정상 범위 최댓값

print(pd.DataFrame({"Q1": q1, "Q3": q3, "IQR": iqr,
                    "정상 최소": min_val, "정상 최대": max_val}).to_string())

> **📝 고친 부분**
> - `q1` `q3` `min_val` `max_val` 을 각각 따로 찍고 대부분 주석으로 막아 둬서 **IQR 하나만** 보였다
>   → 다섯 값을 **한 표**로 모아 서로 비교되게 함
> - 마크다운 필기의 `(Q1 - (1.5-IQR))` · `(Q3 + (1.5+IQR))` 는 **곱하기(`*`)** 가 맞다 (교재 p.30)

### 3-1. 비교 연산자와 논리 연산자 (교재 p.33)

- **비교 연산자** — 두 값의 크기를 비교한다. 결과는 `True` / `False`
- **논리 연산자** — 두 가지 이상의 조건을 **동시에** 처리한다

| 판다스 | 뜻 | 파이썬 |
|--------|-----|--------|
| `&` | 논리곱 (둘 다 참) | `and` |
| `\|` | 논리합 (하나라도 참) | `or` |
| `~` | 논리부정 (참↔거짓 뒤집기) | `not` |

> ⚠️ Series·DataFrame 에는 **`and` `or` `not` 을 쓸 수 없다.**
> 값이 하나가 아니라 여러 개라 "참이냐 거짓이냐"를 하나로 정할 수 없기 때문에 **ValueError** 가 난다.
> 반드시 `&` `|` `~` 를 쓰고, 조건마다 **괄호**로 묶는다.

In [ ]:
# and 를 쓰면 실제로 어떤 에러가 나는지 확인해 보기
try:
    test = (df < min_val) or (df > max_val)
except ValueError as e:
    print("ValueError :", e)

print("-" * 40)

# 올바른 방법 : | (논리합)
outlier = (df < min_val) | (df > max_val)
print("이상치 여부 (bool DataFrame) :")
print(outlier.head())

### 3-2. 이상치 **제거** (교재 p.32 ①)

1. 전체 수치형 컬럼의 Q1 · Q3 · IQR 계산 → (위에서 완료)
2. 컬럼별 정상 범위 최솟값 / 최댓값 계산 → (위에서 완료)
3. 데이터프레임 전체에 논리 연산자를 적용해 **불리언 DataFrame** 생성
4. `any(axis=1)` 로 **어떤 컬럼에서든 이상치가 하나라도 있는 행** 식별
5. `~` 로 뒤집어 **정상인 행만** 불리언 인덱싱 → `reset_index(drop=True)` 로 인덱스 재설정

> `axis=1` 은 **행 방향으로 훑는다**는 뜻이다. 한 행에 `True` 가 하나라도 있으면 그 행은 `True`.

In [ ]:
# 3) 불리언 DataFrame
outlier = (df < min_val) | (df > max_val)

# 4) 행마다 이상치가 하나라도 있나?
has_outlier = outlier.any(axis=1)
print("이상치가 있는 행 :", has_outlier.sum(), "개")

# 5) 뒤집어서 '정상인 행만' True 로
normal_rows = ~has_outlier
print("정상인 행       :", normal_rows.sum(), "개")

print("-" * 40)

# 불리언 인덱싱 + 인덱스 재설정
df_clean = df[normal_rows].reset_index(drop=True)

print("원본 :", df.shape, "-> 이상치 제거 후 :", df_clean.shape)
print(df_clean.head())

> **📝 고친 부분**
> - 변수명 오타 — `hax_outlier` → **`has_outlier`**, `noml_bool` → **`normal_rows`**
> - `print(f"논리 부정{hax_outlier}")` 은 **뒤집기 전(`~` 적용 전)** 값을 "논리 부정"이라는 이름으로 찍고 있었다
>   → 뒤집기 전후를 **개수로** 나란히 출력해 무엇이 달라졌는지 보이게 함
> - `arr = output.values` → `.values` 로 바꾸지 않아도 **Series 그대로 불리언 인덱싱**이 된다 (`df[normal_rows]`)
> - 교재 p.32 의 마지막 단계인 **`reset_index(drop=True)`** 가 빠져 있어 추가
> - 주석으로 남아 있던 `or` 버전은 지우지 않고 3-1 에서 **실제 에러를 띄워** 왜 안 되는지 보이게 함

### 3-3. 이상치 **대체** — `clip()` (교재 p.32 ②)

제거하면 행이 줄어든다. 줄이고 싶지 않을 때는 **범위 밖의 값을 경계값으로 눌러 준다.**

- 정상 범위 **최솟값 미만** → 최솟값으로
- 정상 범위 **최댓값 초과** → **최댓값**으로

```python
df_clipped = df.clip(lower=min_val, upper=max_val, axis=1)
```

In [ ]:
df_clipped = df.clip(lower=min_val, upper=max_val, axis=1)

print("행 개수 :", df.shape[0], "-> clip 후 :", df_clipped.shape[0], "(줄지 않는다)")

print("-" * 40)

print("[clip 전후 최솟값 · 최댓값]")
print(pd.DataFrame({
    "원본 min":  df.min(),   "원본 max":  df.max(),
    "clip min": df_clipped.min(), "clip max": df_clipped.max(),
    "정상 최소": min_val,     "정상 최대": max_val,
}).to_string())

> **📝 고친 부분**
> - 교재 p.32 의 **대체(`clip`) 방법이 필기에 없어서** 새로 추가했다
> - 교재 설명의 "최댓값 초과는 정상 범위 **최솟값**으로 일괄 대체" 는 오타로 보인다.
>   `clip` 은 **최댓값 초과 → 최댓값**으로 누른다.
> - 위 표에서 `hum` 은 최솟값 17 이 정상 최소 25.5 로 눌렸고(`clip min` 25.5),
>   `temp` 는 최댓값 27 이 정상 최대 25 로 눌렸다. 원래 범위 안에 있던 값은 그대로다
>   (`hum` 의 최댓값 100 은 정상 최대 133.5 보다 작아 바뀌지 않았다).

## 4. 오늘 정리

- **누락 데이터** — 개수는 `df.isnull().sum()`, 전체는 `.sum().sum()`. 반복문으로 세지 않는다.
- 지울 땐 `dropna(ignore_index=True)`(행 통째로) · `subset=[...]`(특정 컬럼만), 채울 땐 `fillna(값)`.
- **groupby** — `df.groupby(by='컬럼')['대상'].mean()`, 여러 집계는 `agg(새이름=('원본', '함수'))`.
- **이상치** — `IQR = Q3 - Q1`, 정상 범위는 `Q1 - 1.5*IQR` ~ `Q3 + 1.5*IQR`.
- 판다스 조건에는 `and` `or` `not` 대신 **`&` `|` `~`** 를 쓰고 조건마다 괄호로 묶는다.
- 제거는 `any(axis=1)` → `~` → 불리언 인덱싱 → `reset_index(drop=True)`,
  대체는 `clip(lower=, upper=, axis=1)`.